### Setup

In [1]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
model_name = "distilgpt2"  # small + fast
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

### Helper function to measure latency

In [3]:
def run_inference(prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt")
    
    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    end = time.time()
    
    return end - start

### Run with increasing context lengths

In [4]:
base_sentence = "Artificial intelligence is transforming industries."

In [5]:
for repeat in [1, 5, 10, 20]:
    long_prompt = " ".join([base_sentence] * repeat)
    latency = run_inference(long_prompt)
    print(f"Context repeats: {repeat:>2} | Latency: {latency:.3f} sec")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Context repeats:  1 | Latency: 2.222 sec


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Context repeats:  5 | Latency: 1.175 sec


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Context repeats: 10 | Latency: 1.198 sec
Context repeats: 20 | Latency: 1.297 sec


### Sampling Parameters Impact (Same Context, Different Outputs)

In [6]:
prompt = "AI will change the future of healthcare by"
inputs = tokenizer(prompt, return_tensors="pt")

def generate_with_sampling(temperature, top_p):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=True,
            temperature=temperature,
            top_p=top_p
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [7]:
settings = [
    (0.2, 0.9),
    (0.7, 0.9),
    (1.0, 0.95),
]

for temp, top_p in settings:
    print(f"\nTemperature={temp}, Top-p={top_p}")
    print(generate_with_sampling(temp, top_p).strip())

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Temperature=0.2, Top-p=0.9


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


AI will change the future of healthcare by providing a free and open healthcare system.

Temperature=0.7, Top-p=0.9


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


AI will change the future of healthcare by 2030, and it will be clear that it is not just about the health care system but also about the future of healthcare, which is about health care

Temperature=1.0, Top-p=0.95
AI will change the future of healthcare by a new approach to the healthcare system. With a new approach to the healthcare system, it is already being considered for the role of healthcare providers to lead


### KV-Cache Reuse (The Core Optimization)

In [ ]:
# Step 1: Run full prompt (no cache reuse)
prompt = "Large language models are powerful because"
inputs = tokenizer(prompt, return_tensors="pt")

start = time.time()
with torch.no_grad():
    output = model(
        **inputs,
        use_cache=True
    )
end = time.time()

print(f"Initial forward pass latency: {end - start:.3f} sec")
past_key_values = output.past_key_values

In [ ]:
# Step 2: Continue WITHOUT recomputing prefix
next_token = tokenizer(" they can reason", return_tensors="pt")["input_ids"]

start = time.time()
with torch.no_grad():
    output_cached = model(
        input_ids=next_token,
        past_key_values=past_key_values,
        use_cache=True
    )
end = time.time()

print(f"Cached continuation latency: {end - start:.4f} sec")

In [ ]:
# Step 3: Compare to full recomputation
full_prompt = prompt + " they can reason"
inputs_full = tokenizer(full_prompt, return_tensors="pt")

start = time.time()
with torch.no_grad():
    _ = model(**inputs_full)
end = time.time()

print(f"Full recomputation latency: {end - start:.4f} sec")